# HOGENOM CCP Reconciliation with GPUREC

This notebook loads the AleRax-style HOGENOM CCP family manifest, evaluates it with GPUREC, and saves reconciliation summaries. The default run uses genewise rates from AleRax checkpoint files when they are available.

The main output is a per-family table with GPUREC likelihoods and root-placement posteriors. A later cell computes and saves the full `Pi` matrix for one selected family.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path.cwd()
if not (REPO / "gpurec").exists():
    REPO = Path("/home/enzo/Documents/git/gpurec/gpurec")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from gpurec import GeneReconModel
from gpurec.api.autograd import _extract_parameters
from gpurec.core.forward import Pi_wave_forward
from gpurec.core.likelihood import E_fixed_point, compute_log_likelihood_root_rows
from gpurec.core.model import GeneDataset, parse_alerax_family_file

cu13_lib = Path("/home/enzo/miniforge3/lib/python3.12/site-packages/nvidia/cu13/lib")
if torch.cuda.is_available() and torch.version.cuda == "13.0" and cu13_lib.exists():
    ld_library_path = os.environ.get("LD_LIBRARY_PATH", "")
    if str(cu13_lib) not in ld_library_path:
        print("CUDA 13 NVRTC libraries are not on LD_LIBRARY_PATH.")
        print("If the first GPU cell fails with libnvrtc-builtins.so.13.0, restart Jupyter with:")
        print(f"LD_LIBRARY_PATH={cu13_lib}:$LD_LIBRARY_PATH jupyter lab")

print("repo", REPO)
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.version.cuda)

## Configuration

For a smoke test, keep `MAX_FAMILIES` small. Set it to `None` for all families with checkpoint rates. `FIXED_ITERS_E` and `FIXED_ITERS_PI` are the GPUREC evaluator settings; AleRax's current CLI has one shared `--dtl-iterations` knob, but GPUREC lets you separate them here.

In [ ]:
HOGENOM_DIR = REPO / "tests" / "data" / "HOGENOM" / "hogenom"
FAMILIES_FILE = HOGENOM_DIR / "hogenom_families.local.txt"
ALERAX_OUTPUT = HOGENOM_DIR / "output_alerax_corrected"
ALERAX_CHECKPOINT_DIR = ALERAX_OUTPUT / "checkpoint"
ALERAX_PER_FAM_LIKELIHOODS = ALERAX_OUTPUT / "per_fam_likelihoods.txt"

inferred_species_tree = ALERAX_OUTPUT / "species_trees" / "inferred_species_tree.newick"
SPECIES_TREE = inferred_species_tree if inferred_species_tree.exists() else HOGENOM_DIR / "hogenom_S.tree"

OUTPUT_DIR = HOGENOM_DIR / "output_gpurec_ccp_reconciliation"
PREPROCESS_CACHE_DIR = OUTPUT_DIR / "preprocess_cache"

DEVICE = "cuda"
DTYPE = torch.float64
MODE = "genewise"  # "genewise", "global", or "specieswise"

START_FAMILY = 0
MAX_FAMILIES = 24  # Set to None for all checkpointed HOGENOM families.
CHUNK_SIZE = 24

USE_ALERAX_CHECKPOINT_RATES = True
DEFAULT_RATES = (0.1, 0.1, 0.1)

FIXED_ITERS_E = 16
FIXED_ITERS_PI = 6
NEUMANN_TERMS = 6
MAX_WAVE_SIZE = 32768
USE_PRUNING = True

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but torch.cuda.is_available() is false")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("families", FAMILIES_FILE)
print("species_tree", SPECIES_TREE)
print("output", OUTPUT_DIR)

In [ ]:
def read_alerax_reference_likelihoods(path: Path) -> dict[str, float]:
    if not path.exists():
        return {}
    out: dict[str, float] = {}
    for raw in path.read_text().splitlines():
        parts = raw.split()
        if len(parts) >= 2:
            out[parts[0]] = float(parts[1])
    return out


def read_checkpoint_rates(path: Path) -> tuple[float, float, float]:
    lines = path.read_text().splitlines()
    if len(lines) < 2:
        raise ValueError(f"checkpoint file has no rate line: {path}")
    values = [float(x) for x in lines[1].split()]
    if len(values) < 3:
        raise ValueError(f"checkpoint rate line has fewer than 3 values: {path}")
    return values[0], values[1], values[2]


def posterior_from_log2_rows(log_rows: torch.Tensor) -> torch.Tensor:
    shifted = log_rows - log_rows.max(dim=1, keepdim=True).values
    probs = torch.exp2(shifted)
    return probs / probs.sum(dim=1, keepdim=True)


all_family_names, all_tree_paths, all_leaf_maps = parse_alerax_family_file(FAMILIES_FILE)
tree_paths_by_name = dict(zip(all_family_names, all_tree_paths))
leaf_maps_by_name = dict(zip(all_family_names, all_leaf_maps))
family_rank_by_name = {name: i for i, name in enumerate(all_family_names)}

alerax_loglik_by_name = read_alerax_reference_likelihoods(ALERAX_PER_FAM_LIKELIHOODS)
rates_by_name: dict[str, tuple[float, float, float]] = {}
if ALERAX_CHECKPOINT_DIR.exists():
    for path in sorted(ALERAX_CHECKPOINT_DIR.glob("*.txt")):
        if path.stem not in family_rank_by_name:
            continue
        rates_by_name[path.stem] = read_checkpoint_rates(path)

available_names = list(all_family_names)
if USE_ALERAX_CHECKPOINT_RATES:
    available_names = [name for name in available_names if name in rates_by_name]

selected_names = available_names[START_FAMILY:]
if MAX_FAMILIES is not None:
    selected_names = selected_names[:MAX_FAMILIES]

print("families in manifest", len(all_family_names))
print("families with AleRax checkpoint rates", len(rates_by_name))
print("families with AleRax reference likelihoods", len(alerax_loglik_by_name))
print("selected families", len(selected_names))
selected_names[:5]

In [ ]:
def build_model_for_names(chunk_names: list[str]) -> GeneReconModel:
    device = torch.device(DEVICE)
    genewise = MODE == "genewise"
    specieswise = MODE == "specieswise"
    chunk_tree_paths = [tree_paths_by_name[name] for name in chunk_names]
    chunk_leaf_maps = [leaf_maps_by_name[name] for name in chunk_names]

    dataset = GeneDataset(
        species_tree_path=str(SPECIES_TREE),
        gene_tree_paths=chunk_tree_paths,
        genewise=genewise,
        specieswise=specieswise,
        dtype=DTYPE,
        device=device,
        preprocess_cache_dir=PREPROCESS_CACHE_DIR,
        family_names=chunk_names,
        leaf_species_maps=chunk_leaf_maps,
    )

    if MODE == "genewise":
        rates = [rates_by_name.get(name, DEFAULT_RATES) for name in chunk_names]
        theta_init = torch.log2(torch.tensor(rates, dtype=DTYPE, device=device))
    elif MODE == "specieswise":
        base = torch.log2(torch.tensor(DEFAULT_RATES, dtype=DTYPE, device=device))
        theta_init = base.unsqueeze(0).expand(int(dataset.S), -1).clone()
    else:
        theta_init = torch.log2(torch.tensor(DEFAULT_RATES, dtype=DTYPE, device=device))

    return GeneReconModel(
        dataset=dataset,
        mode=MODE,
        theta_init=theta_init,
        fixed_iters_E=FIXED_ITERS_E,
        fixed_iters_Pi=FIXED_ITERS_PI,
        neumann_terms=NEUMANN_TERMS,
        max_wave_size=MAX_WAVE_SIZE,
        use_pruning=USE_PRUNING,
    )


@torch.no_grad()
def evaluate_root_reconciliation(model: GeneReconModel) -> tuple[torch.Tensor, torch.Tensor]:
    static = model.static
    log_pS, log_pD, log_pL, max_transfer_vec = _extract_parameters(model.theta.detach(), static)
    e_max_iters = static.fixed_iters_E if static.fixed_iters_E is not None else static.max_iters_E
    e_tolerance = -1.0 if static.fixed_iters_E is not None else static.tol_E
    E_out = E_fixed_point(
        species_helpers=static.species_helpers,
        log_pS=log_pS,
        log_pD=log_pD,
        log_pL=log_pL,
        max_transfer_mat=max_transfer_vec,
        max_iters=e_max_iters,
        tolerance=e_tolerance,
        warm_start_E=None,
        dtype=static.dtype,
        device=static.device,
        ancestors_T=static.ancestors_T,
    )
    Pi_out = Pi_wave_forward(
        wave_layout=static.wave_layout,
        species_helpers=static.species_helpers,
        E=E_out["E"],
        Ebar=E_out["E_bar"],
        E_s1=E_out["E_s1"],
        E_s2=E_out["E_s2"],
        log_pS=log_pS,
        log_pD=log_pD,
        max_transfer_mat=max_transfer_vec,
        device=static.device,
        dtype=static.dtype,
        fixed_iters=static.fixed_iters_Pi,
        return_original=False,
        return_root_rows=True,
        family_idx=static.wave_layout.get("family_idx") if static.genewise else None,
    )
    root_log2 = Pi_out["Pi_root_rows"]
    nll_bits = compute_log_likelihood_root_rows(root_log2, E_out["E"])
    root_probs = posterior_from_log2_rows(root_log2)
    return nll_bits.detach(), root_probs.detach()


def chunked(iterable: list[str], size: int):
    for start in range(0, len(iterable), size):
        yield start, iterable[start:start + size]

## Run Reconciliation

This evaluates chunks independently so the full HOGENOM CCP set can run without keeping the whole `Pi` state resident at once.

In [ ]:
rows: list[dict[str, object]] = []
top_rows: list[dict[str, object]] = []
ln2 = math.log(2.0)
t0 = time.perf_counter()

for chunk_index, (start, chunk_names) in enumerate(chunked(selected_names, CHUNK_SIZE)):
    model = build_model_for_names(chunk_names)
    nll_bits, root_probs = evaluate_root_reconciliation(model)
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    species_names = model.static.species_helpers["names"]
    top_prob, top_idx = torch.topk(root_probs, k=min(5, root_probs.shape[1]), dim=1)
    nll_bits_cpu = nll_bits.cpu().numpy()
    top_prob_cpu = top_prob.cpu().numpy()
    top_idx_cpu = top_idx.cpu().numpy()

    for local_i, name in enumerate(chunk_names):
        rates = rates_by_name.get(name, DEFAULT_RATES)
        gpurec_loglik_nats = -float(nll_bits_cpu[local_i]) * ln2
        ref = alerax_loglik_by_name.get(name)
        rows.append({
            "selection_rank": start + local_i,
            "family_file_rank": family_rank_by_name[name],
            "family": name,
            "D": rates[0],
            "L": rates[1],
            "T": rates[2],
            "gpurec_nll_bits": float(nll_bits_cpu[local_i]),
            "gpurec_loglik_nats": gpurec_loglik_nats,
            "alerax_loglik_nats": ref,
            "diff_nats": None if ref is None else gpurec_loglik_nats - float(ref),
            "top_root_species": species_names[int(top_idx_cpu[local_i, 0])],
            "top_root_species_index": int(top_idx_cpu[local_i, 0]),
            "top_root_probability": float(top_prob_cpu[local_i, 0]),
            "chunk": chunk_index,
        })
        for rank in range(top_prob_cpu.shape[1]):
            sp_i = int(top_idx_cpu[local_i, rank])
            top_rows.append({
                "family": name,
                "rank": rank + 1,
                "species_index": sp_i,
                "species": species_names[sp_i],
                "probability": float(top_prob_cpu[local_i, rank]),
            })

    del model, nll_bits, root_probs
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elapsed = time.perf_counter() - t0
    print(f"chunk {chunk_index + 1}/{math.ceil(len(selected_names) / CHUNK_SIZE)} families={len(rows)}/{len(selected_names)} elapsed_s={elapsed:.1f}")

family_df = pd.DataFrame(rows)
top_root_df = pd.DataFrame(top_rows)
family_df.to_csv(OUTPUT_DIR / "gpurec_hogenom_ccp_family_root_summary.csv", index=False)
top_root_df.to_csv(OUTPUT_DIR / "gpurec_hogenom_ccp_root_top5.csv", index=False)

summary = {
    "families": int(len(family_df)),
    "elapsed_s": float(time.perf_counter() - t0),
    "mode": MODE,
    "device": DEVICE,
    "dtype": str(DTYPE).replace("torch.", ""),
    "fixed_iters_E": FIXED_ITERS_E,
    "fixed_iters_Pi": FIXED_ITERS_PI,
    "chunk_size": CHUNK_SIZE,
    "species_tree": str(SPECIES_TREE),
    "families_file": str(FAMILIES_FILE),
    "total_gpurec_loglik_nats": float(family_df["gpurec_loglik_nats"].sum()),
}
if family_df["alerax_loglik_nats"].notna().any():
    valid = family_df.dropna(subset=["alerax_loglik_nats", "diff_nats"])
    summary.update({
        "families_with_alerax_reference": int(len(valid)),
        "total_alerax_loglik_nats": float(valid["alerax_loglik_nats"].sum()),
        "total_diff_nats": float(valid["diff_nats"].sum()),
        "mean_abs_diff_nats": float(valid["diff_nats"].abs().mean()),
        "max_abs_diff_nats": float(valid["diff_nats"].abs().max()),
    })

(OUTPUT_DIR / "gpurec_hogenom_ccp_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
summary

In [ ]:
display(family_df.head(20))
display(top_root_df.head(20))
print("saved", OUTPUT_DIR / "gpurec_hogenom_ccp_family_root_summary.csv")
print("saved", OUTPUT_DIR / "gpurec_hogenom_ccp_root_top5.csv")
print("saved", OUTPUT_DIR / "gpurec_hogenom_ccp_summary.json")

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

axes[0].hist(family_df["gpurec_nll_bits"], bins=60, color="#4477AA", alpha=0.85)
axes[0].set_title("GPUREC per-family NLL")
axes[0].set_xlabel("NLL (bits)")
axes[0].set_ylabel("Families")

axes[1].hist(family_df["top_root_probability"], bins=60, color="#228833", alpha=0.85)
axes[1].set_title("Top root placement posterior")
axes[1].set_xlabel("posterior probability")
axes[1].set_ylabel("Families")

fig.savefig(OUTPUT_DIR / "gpurec_hogenom_ccp_reconciliation_summary.png", dpi=160)
plt.show()

if family_df["alerax_loglik_nats"].notna().any():
    valid = family_df.dropna(subset=["alerax_loglik_nats", "diff_nats"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    axes[0].scatter(valid["alerax_loglik_nats"], valid["gpurec_loglik_nats"], s=10, alpha=0.65)
    lo = min(valid["alerax_loglik_nats"].min(), valid["gpurec_loglik_nats"].min())
    hi = max(valid["alerax_loglik_nats"].max(), valid["gpurec_loglik_nats"].max())
    axes[0].plot([lo, hi], [lo, hi], color="black", linewidth=1)
    axes[0].set_title("GPUREC vs AleRax likelihood")
    axes[0].set_xlabel("AleRax log-likelihood (nats)")
    axes[0].set_ylabel("GPUREC log-likelihood (nats)")
    axes[1].hist(valid["diff_nats"], bins=60, color="#CC6677", alpha=0.85)
    axes[1].axvline(0.0, color="black", linewidth=1)
    axes[1].set_title("GPUREC - AleRax")
    axes[1].set_xlabel("difference (nats)")
    axes[1].set_ylabel("Families")
    fig.savefig(OUTPUT_DIR / "gpurec_hogenom_ccp_alerax_comparison.png", dpi=160)
    plt.show()

## Materialize a Full Pi Matrix for One Family

`Pi` is saved in log2 space with shape `[clades, species]`. The row order is the original clade order for that family. This is useful for inspecting more than the root posterior without materializing the full HOGENOM matrix.

In [ ]:
PI_FAMILY_NAME = selected_names[0]

pi_model = build_model_for_names([PI_FAMILY_NAME])
with torch.no_grad():
    pi_log2 = pi_model.pi_matrix(original_order=True).detach().cpu().numpy()
species_names = np.array(pi_model.static.species_helpers["names"], dtype=object)
rates = rates_by_name.get(PI_FAMILY_NAME, DEFAULT_RATES)

pi_path = OUTPUT_DIR / f"{PI_FAMILY_NAME}.gpurec_pi_log2.npz"
np.savez_compressed(
    pi_path,
    family=np.array([PI_FAMILY_NAME], dtype=object),
    species_names=species_names,
    pi_log2=pi_log2,
    rates=np.array(rates, dtype=float),
)
print("Pi shape", pi_log2.shape)
print("saved", pi_path)

root_id = int(pi_model._dataset.families[0]["root_clade_id"])
root_probs = posterior_from_log2_rows(torch.from_numpy(pi_log2[root_id:root_id + 1])).numpy()[0]
top = np.argsort(root_probs)[::-1][:10]
pd.DataFrame({
    "rank": np.arange(1, len(top) + 1),
    "species_index": top,
    "species": species_names[top],
    "root_probability": root_probs[top],
})